In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==========================================
# 1. PARAMETERIZATION (Databricks Widget)
# ==========================================
dbutils.widgets.text("batch_name", "batch_1", "Target Batch Name")
batch_name = dbutils.widgets.get("batch_name")

print(f"Processing Silver Customers (Dimension) for Batch: {batch_name}")

# ==========================================
# 2. EXTRACT INCOMING DATA FROM BRONZE
# ==========================================
df_bronze_raw = spark.read.table(
    "ecommerce_lakehouse.bronze.bronze_olist_customers"
).filter(F.col("_batch_id") == batch_name)

# ==========================================
# 3. TRANSFORMATIONS & DATA HYGIENE
# ==========================================
# Clean columns and drop customer_id (keeping only unique human identity)
df_cleaned = df_bronze_raw.select(
    F.trim(F.col("customer_unique_id")).alias("customer_unique_id"),
    F.coalesce(
        F.trim(F.col("customer_zip_code_prefix")), F.lit("00000")
    ).alias("postal_code"),
    F.initcap(F.trim(F.col("customer_city"))).alias("city"),
    F.upper(F.trim(F.col("customer_state"))).alias("state"),
).dropDuplicates(
    ["customer_unique_id"]
)  # Ensure 1 record per unique human per batch

# Compute Row Hash using SHA-256 for change detection (Excluding keys)
df_hashed = df_cleaned.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "||",
            F.coalesce(F.col("postal_code"), F.lit("")),
            F.coalesce(F.col("city"), F.lit("")),
            F.coalesce(F.col("state"), F.lit("")),
        ),
        256,
    ),
)

# ==========================================
# 4. TARGET TABLE CHECK & INITIALIZATION
# ==========================================
target_table_name = "ecommerce_lakehouse.silver.silver_customers"
table_exists = spark.catalog.tableExists(target_table_name)
current_timestamp = F.current_timestamp()

if not table_exists:
  print("Target table does not exist. Initializing SCD Type 2 dimension...")

  # Generate surrogate keys (C000001, C000002...) for initial load
  window_spec = Window.orderBy("customer_unique_id")
  df_transformed = df_hashed.withColumn(
      "customer_key",
      F.concat(
          F.lit("C"),
          F.lpad(
              (F.row_number().over(window_spec)).cast("string"), 6, "0"
          ),
      ),
  ).select(
      "customer_key",
      "customer_unique_id",
      "postal_code",
      "city",
      "state",
      "row_hash",
      F.lit(True).alias("is_current"),
      current_timestamp.alias("effective_start_date"),
      F.lit("9999-12-31 23:59:59").cast("timestamp").alias(
          "effective_end_date"
      ),
      current_timestamp.alias("_etl_inserted_at"),
      current_timestamp.alias("_etl_updated_at"),
      F.lit(False).alias("_delete_flag"),
  )

  # Write initial table
  df_transformed.write.format("delta").mode("overwrite").saveAsTable(
      target_table_name
  )
  print("Successfully initialized silver_customers dimension!")

else:
  print("Table exists. Executing SCD Type 2 Upsert Merge...")

  # Find max surrogate key offset to avoid duplicate keys on incremental loads
  max_key_row = spark.sql(
      f"SELECT MAX(CAST(SUBSTRING(customer_key, 2) AS INT)) as max_key FROM"
      f" {target_table_name}"
  ).collect()[0]
  max_key_val = (
      max_key_row["max_key"] if max_key_row["max_key"] is not None else 0
  )

  # Assign new sequential surrogate keys for incoming data
  df_incoming = df_hashed.withColumn(
      "temp_seq", F.row_number().over(Window.orderBy("customer_unique_id"))
  ).withColumn(
      "new_customer_key",
      F.concat(
          F.lit("C"),
          F.lpad(
              (F.col("temp_seq") + F.lit(max_key_val)).cast("string"), 6, "0"
          ),
      ),
  )

  deltaTable = DeltaTable.forName(spark, target_table_name)

  # ==========================================
  # 5. DELTA LAKE SCD TYPE 2 MERGE LOGIC
  # ==========================================
  # Action A: Expire old records if attributes (row_hash) changed
  deltaTable.alias("target").merge(
      source=df_incoming.alias("source"),
      condition=(
          "target.customer_unique_id = source.customer_unique_id AND"
          " target.is_current = true"
      ),
  ).whenMatchedUpdate(
      condition="target.row_hash <> source.row_hash",
      set={
          "is_current": F.lit(False),
          "effective_end_date": current_timestamp,
          "_etl_updated_at": current_timestamp,
      },
  ).execute()

  # Action B: Isolate brand-new customers and new versions of updated customers
  df_inserts = df_incoming.join(
      spark.table(target_table_name).filter("is_current = true").select(
          "customer_unique_id", "row_hash"
      ),
      on="customer_unique_id",
      how="left_anti",
  )

  if df_inserts.count() > 0:
    df_final_inserts = df_inserts.select(
        F.col("new_customer_key").alias("customer_key"),
        "customer_unique_id",
        "postal_code",
        "city",
        "state",
        "row_hash",
        F.lit(True).alias("is_current"),
        current_timestamp.alias("effective_start_date"),
        F.lit("9999-12-31 23:59:59").cast("timestamp").alias(
            "effective_end_date"
        ),
        current_timestamp.alias("_etl_inserted_at"),
        current_timestamp.alias("_etl_updated_at"),
        F.lit(False).alias("_delete_flag"),
    )

    df_final_inserts.write.format("delta").mode("append").saveAsTable(
        target_table_name
    )
    print(
        f"Inserted {df_final_inserts.count()} active records (new & updated)."
    )
  else:
    print("No new active rows to insert.")

print("Silver Customers dimension processing completed successfully!")

Processing Silver Customers (Dimension) for Batch: batch_1
Target table does not exist. Initializing SCD Type 2 dimension...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Successfully initialized silver_customers dimension!
Silver Customers dimension processing completed successfully!
